1. Import Libraries

In [6]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

2. Load Dataset

In [9]:
df = pd.read_csv("IMDB Dataset.csv")  # change file path if needed

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


3. Data Understanding

In [10]:
print("Shape:", df.shape)
print("Null values:\n", df.isnull().sum())

# Convert labels to numeric
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

Shape: (50000, 2)
Null values:
 review       0
sentiment    0
dtype: int64


4. NLP Preprocessing



In [11]:
# Create reusable preprocessing function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-zA-Z]', ' ', text)      # remove special chars

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [12]:
# Apply preprocessing
df['cleaned_text'] = df['review'].apply(preprocess_text)

5. Feature Engineering


In [13]:
# Bag of Words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_text'])

In [15]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_text'])

In [17]:
# Target Variable
y = df['sentiment']

6. Train-Test Split

In [18]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

7. Model Building

In [19]:
# Function to Evaluate Models
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [20]:
# 1. Logistic Regression
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

print("Logistic Regression (TF-IDF)")
evaluate_model(lr, X_test_tfidf, y_test)

Logistic Regression (TF-IDF)
Accuracy: 0.8893
Precision: 0.8798299845440495
Recall: 0.9037507441952768
F1 Score: 0.8916299559471366

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.87      0.89      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [23]:
# 2. Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

print("Naive Bayes (BoW)")
evaluate_model(nb, X_test_bow, y_test)

Naive Bayes (BoW)
Accuracy: 0.8501
Precision: 0.8542834267413931
Recall: 0.8469934510815638
F1 Score: 0.8506228201295466

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.85      0.85      4961
           1       0.85      0.85      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [24]:
# 3. Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)

print("Decision Tree (BoW)")
evaluate_model(dt, X_test_bow, y_test)

Decision Tree (BoW)
Accuracy: 0.714
Precision: 0.7204127048351203
Recall: 0.7066878348878746
F1 Score: 0.7134842716890403

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.72      0.71      4961
           1       0.72      0.71      0.71      5039

    accuracy                           0.71     10000
   macro avg       0.71      0.71      0.71     10000
weighted avg       0.71      0.71      0.71     10000



8. Comparison & Insights

### Model Comparison

| Model               | Vectorizer | Accuracy |
|--------------------|------------|----------|
| Logistic Regression| TF-IDF     | High     |
| Naive Bayes        | BoW        | Medium   |
| Decision Tree      | BoW        | Lower    |

### Insights:
- TF-IDF performed better than Bag of Words.
- Logistic Regression gave the best results due to handling high-dimensional data well.
- Naive Bayes is fast but less accurate.
- Decision Tree tends to overfit on text data.

### Conclusion:
Best combination: TF-IDF + Logistic Regression

9. Final Pipeline Flow

Raw Text  
→ Text Cleaning (lowercase, stopwords removal, lemmatization)  
→ Feature Extraction (BoW / TF-IDF)  
→ Model Training (LR, NB, DT)  
→ Evaluation  
→ Comparison  